In [70]:
import sys 
import xarray as xr
import numpy as np
import os
import pandas as pd
import yaml
from numpy import random
import math
from datetime import datetime, timedelta
from parcels import FieldSet, Field, VectorField, ParticleSet, JITParticle, ParcelsRandom, Variable,StatusCode
from glob import glob

import matplotlib.pyplot as plt

from parcels import ScipyParticle, AdvectionRK4,AdvectionRK4_3D


In [71]:
sys.path.append(os.getcwd()+'/Source_vvl')

from OP_Kernels import *
from OP_functions import *

In [72]:
jjii = xr.open_dataset('~/MOAD/grid/grid_from_lat_lon_mask999.nc')

def finder(jjii,lati,loni):
    j = [jjii.jj.sel(lats=lati, lons=loni, method='nearest').item()][0]
    i = [jjii.ii.sel(lats=lati, lons=loni, method='nearest').item()][0]
    return j,i

finder(jjii,49.35844243668596,-123.80204117521916)

(512, 251)

In [73]:
def load_config(config_yaml):
   with open(config_yaml) as f:
       config = yaml.safe_load(f)
   return config

In [74]:
def pandas_deploy(N,MFc,r,dd,dtp):
    n =1
    MFc = float(MFc)
    Outfall_deploy = pd.read_csv(N, index_col = [0])
    Pol = list(Outfall_deploy.Population)
    Lat = Outfall_deploy.Lat
    Lon = Outfall_deploy.Lon
    Depth = Outfall_deploy.Depth
    clat = []
    clon = []
    cz = []
    for i,loc in enumerate(Pol):
        for j in range(int(round((loc*250*dtp)/MFc,0))):
            clat.append(Lat.iat[i])
            clon.append(Lon.iat[i])
            try:
                cz.append(float(Depth.iat[i]))
            except ValueError: #This is for rivers
                cz.append(2)
    N = len(clat)
    deg2m = 111319.5 * np.cos(49 * np.pi / 180)
    var = (r / (deg2m * 3))**2
    x_offset, y_offset = np.random.multivariate_normal([0, 0], [[var, 0], [0, var]], [n,N]).T #Here we define a cloud of deployment points.
    z_offset=np.random.random_sample([N]).T*(dd) #add some depth variability
    lon = np.zeros([N])
    lat = np.zeros([N])
    z = np.zeros([N])
    for i in range(N):
        lon[i]=(clon[i] + x_offset[i])
        lat[i]=(clat[i] + y_offset[i])
        z[i]=(cz[i] + z_offset[i])
    return lon,lat,z

In [75]:
coord = xr.open_dataset(path('salish')['coords'],decode_times=False)
mask = xr.open_dataset(path('salish')['mask'],decode_times=False)

In [76]:
def p_unidist(lat0,lon0,bat,dy,dx):
    latbat = np.zeros(bat.shape)
    lonbat = np.zeros(bat.shape)

    for i in range(bat.shape[0]):
        for j in range(bat.shape[1]):
            if bat[i,j]>1e-5:
                latbat[i,j] = lat0[i,j]
                lonbat[i,j] = lon0[i,j]

    yi = np.arange(0,latbat.shape[0],dy)
    xi = np.arange(0,latbat.shape[1],dx)
    plat0,plon0 = latbat[yi,:],lonbat[yi,:]
    plat1,plon1 = plat0[:,xi],plon0[:,xi]

    plon,plat = [],[]
    for i in range(plat1.shape[0]):
        for j in range(plat1.shape[1]):
            if plat1[i,j]>1e-5:
                plat.append(plat1[i,j])
                plon.append(plon1[i,j])
    return plat,plon

In [77]:
#clat,clon = p_unidist(coord.nav_lat,coord.nav_lon,mask.totaldepth,20,20)
# np.save('lat.npy',np.array(clat))
# np.save('lon.npy',np.array(clon))

In [78]:
clat = np.load('lat.npy')
clon = np.load('lon.npy')

In [79]:
cz = np.ones(len(clat))

In [80]:
param = load_config('yaml/config.yaml')
start = datetime(param['startdate']['year'], param['startdate']['month'], param['startdate']['day']) #Start date
length = 1#param['param']['length'] # Set Time length [days] 
dt = param['param']['dt'] #toggle between - or + to pick backwards or forwards 
N = param['param']['N'] # Name of deploy loc file
dd = param['param']['dd'] #max depth difference z in N
name = param['file']['name'] #name output file
dtp = param['param']['dtp'] #how often particle released in hours
odt = param['param']['odt'] #how often data is recorded
rrr = param['param']['r'] #radious of particle deployment
SI = param['param']['SI'] #how many particles per super-individual

length = 1
duration = timedelta(days=length)
print(f"The model will run for {duration.days} days, starting at {start}")

n = 1 
N = len(clat)
deg2m = 111319.5 * np.cos(49 * np.pi / 180)
var = (rrr / (deg2m * 3))**2
x_offset, y_offset = np.random.multivariate_normal([0, 0], [[var, 0], [0, var]], [n,N]).T #Here we define a cloud of deployment points.
z_offset=np.random.random_sample([N]).T*(dd) #add some depth variability
lon = np.zeros([N])
lat = np.zeros([N])
z = np.zeros([N])
for i in range(N):
    lon[i]=(clon[i] + x_offset[i])
    lat[i]=(clat[i] + y_offset[i])
    z[i]=(cz[i] + z_offset[i])


print(f"The model will release {N} particle every timestep")

daterange = [start+timedelta(days=i) for i in range(length)]
fn =  name + '.zarr'
outfile = os.path.join(os.getcwd()+'/results/', fn)

print(f"Output file: {outfile}")

The model will run for 1 days, starting at 2007-04-01 00:00:00
The model will release 217 particle every timestep
Output file: /ocean/jvalenti/MOAD/analysis-jose/notebooks/MP_model/results/test_reseed.zarr


/tmp/ipykernel_4021777/492534788.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lon[i]=(clon[i] + x_offset[i])
/tmp/ipykernel_4021777/492534788.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lat[i]=(clat[i] + y_offset[i])


In [81]:
#Define our Parcels Particles
class MPParticle(JITParticle):    
    fact = Variable('fact', initial = 1)
    status = Variable('status', initial = 1) #0=non initialized, 1=active, 2=beached, 3 = sedimented, 10=lost

In [82]:
cs = {
        "time": ("time", 1),
        "depth": ("depth", 40),   
        "lat": ("latitude", 100),
        "lon": ("longitude", 100),
    }

In [83]:
duration = timedelta(days=length)
paths = path('salish')
cgrid_dim = {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'}
Ulist, Vlist, Wlist, Tlist = [], [], [], []

for day in range(duration.days):
  path_NEMO = make_prefix(start + timedelta(days=day),paths['NEMO'])
  Ulist.append(path_NEMO + '_grid_U.nc')
  Vlist.append(path_NEMO + '_grid_V.nc')
  Wlist.append(path_NEMO + '_grid_W.nc')
  Tlist.append(path_NEMO + '_grid_T.nc')

filenames = {
  'U': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Ulist},
  'V': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Vlist},
  'W': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Wlist}
}
dimensions = {
  'U': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
  'V': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
  'W': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
}
variables = {
  'U': 'vozocrtx',
  'V': 'vomecrty',
  'W': 'vovecrtz'
}

field_set = FieldSet.from_nemo(
    filenames,
    variables,
    dimensions,
    allow_time_extrapolation=True,
    chunksize=cs
)

filenames['totaldepth'] = {'lon': paths['coords'], 'lat': paths['coords'], 'data': paths['mask']}
filenames['ssh'] = {'lon': paths['coords'], 'lat': paths['coords'], 'data': Tlist}
variables['totaldepth']= 'totaldepth'
variables['ssh']= 'sossheig'

#Add Bathymetry 2D field
dimensions = {'lon': 'glamt', 'lat': 'gphit'}
TD = Field.from_netcdf(filenames['totaldepth'], variables['totaldepth'], dimensions,allow_time_extrapolation=True)
field_set.add_field(TD)

#Add SSH 
dimensions = {'lon': 'glamt', 'lat': 'gphit','time': 'time_counter'}
SSH = Field.from_netcdf(filenames['ssh'], variables['ssh'], dimensions,allow_time_extrapolation=True)
field_set.add_field(SSH)

/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might b

## Define constants

In [84]:
field_set.add_constant('deg2met', 111319.5)
field_set.add_constant('latT', 0.6495)   #Average value for SoG #cos(particle.lat*(math.pi/180))
field_set.add_constant('Kh', 0.5)   
field_set.add_constant('SDD', 3.34e-6) #std Diameter
field_set.add_constant('SDL', 24.4e-5) #std length

## Define Sources

In [85]:
lat = np.ones(len(clat))*49.33451996322204
lon = np.ones(len(clat))*-123.83734250024288
z = np.ones(len(clat))*0.01

In [86]:
pset = ParticleSet.from_list(field_set, MPParticle, lon=lon, lat=lat, depth=z, lonlatdepth_dtype=np.float32)

In [87]:
def Advection(particle, fieldset, time): 
    if particle.status == 1: #If particle is not active, do not move it
        ssh = fieldset.sossheig[time, particle.depth, particle.lat, particle.lon] #SSH(t)
        sshn = fieldset.sossheig[time+particle.dt, particle.depth, particle.lat, particle.lon] #SSH(t+dt)
        td = fieldset.totaldepth[time, particle.depth, particle.lat, particle.lon]#Total_depth
        particle.fact = (1+ssh/td) #VVl conversion factor
        VVL = (sshn-ssh)*particle.depth/td #VVl w velocity correction
        #4th order RK
        (u1, v1, w1) = fieldset.UVW[particle]
        lon1 = particle.lon + u1*.5*particle.dt
        lat1 = particle.lat + v1*.5*particle.dt
        dep1 = particle.depth + w1*.5*particle.dt/particle.fact
        (u2, v2, w2) = fieldset.UVW[time + .5 * particle.dt, dep1, lat1, lon1,particle]
        lon2 = particle.lon + u2*.5*particle.dt
        lat2 = particle.lat + v2*.5*particle.dt
        dep2 = particle.depth + w2*.5*particle.dt/particle.fact
        (u3, v3, w3) = fieldset.UVW[time + .5 * particle.dt, dep2, lat2, lon2,particle]
        lon3 = particle.lon + u3*particle.dt
        lat3 = particle.lat + v3*particle.dt
        dep3 = particle.depth + w3*particle.dt/particle.fact
        (u4, v4, w4) = fieldset.UVW[time + particle.dt, dep3, lat3, lon3,particle]
        particle_dlon += (u1 + 2*u2 + 2*u3 + u4) / 6. * particle.dt
        particle_dlat += (v1 + 2*v2 + 2*v3 + v4) / 6. * particle.dt
        particle_ddepth += (w1 + 2*w2 + 2*w3 + w4) /6. * particle.dt/particle.fact + VVL
        if particle_ddepth + particle.depth < 0:
            particle_ddepth = -(2*particle.depth + particle_ddepth) #reflection on surface

In [88]:
def CheckOutOfBounds(particle, fieldset, time):
    if particle.state == StatusCode.ErrorOutOfBounds:
        # if Sbh == 0: #Ready to beach particles
        #     print('particle beached')
        #     particle.state = StatusCode.Success
        #     particle.status = 2 #beached DO NOT MOVE outside domain
        # elif Swh ==0:
        #     print('particle hit wall') #Do not cross wall Keep particle in place
        #     particle.state = StatusCode.Success
        # else:
            print('lost Particle')
            particle.status = 10 #lost
            particle.delete()
        
def KeepInOcean(particle, fieldset, time):
    if particle.state == StatusCode.ErrorThroughSurface:
        particle.depth = 0.0
        particle.state = StatusCode.Success

def export(particle,fieldset,time):
    if particle.status==1:
        Pacific =  -particle.lat*0.84120957 -83.98027258 #Checking if particle gets too close to boundary JdF
        South = particle.lon*0.35157547 +90.26497859 #Checking southern boundary model
        if particle.lon<Pacific:
            particle.status = 5 #Exported to Pacific Ocean
        if particle.lat<South:
            if particle.depth < 5:
                particle.status =2 #Beached
            else:
                particle_dlat += 0.1*particle.dt/fieldset.deg2met 
                #Northward velocity to avoid particles from getting stuck on the wall only an issue at southernmost cell of SSC

In [89]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="The specified chunks separate the stored chunks",
    category=UserWarning,
)

In [90]:
pset.execute(
    [Advection,export,KeepInOcean,CheckOutOfBounds],
    runtime=timedelta(hours=12),
    dt=timedelta(seconds=60),
    output_file=pset.ParticleFile(name='/home/jvalenti/MOAD/results/output.zarr', outputdt=timedelta(minutes=30)))

INFO: Output files are stored in /home/jvalenti/MOAD/results/output.zarr.
100%|██████████| 43200.0/43200.0 [00:41<00:00, 1029.87it/s]
